In [5]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

data_path = "../../datasets/airline_passengers/airline-passengers.csv"

In [ ]:
data = pd.read_csv(data_path).to_numpy()

train_raw, test_raw = data[:120, -1], data[120:, -1]

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
scaled_all = scaler.transform(data[:, -1].reshape(-1, 1)).flatten()



In [ ]:
from numpy import float32


inputs = []
targets = []

for i in range(132):
    inputs.append(scaled_all[i:i+12])
    targets.append(scaled_all[i+12])

X, y = np.array(inputs), np.array(targets)

X = X[:, :, np.newaxis]

X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

X_train, X_test = X[:108], X[108:]
y_train, y_test = y[:108], y[108:]


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=50, num_layers=1, batch_first=True)
        self.output = nn.Linear(50, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.output(out)

        return out



In [ ]:
model = LSTMModel()
model.train()


optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.MSELoss() 

loss_list = []

for epoch in range(1000):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    loss_list.append(loss.item())


    if epoch % 100 == 0:
        print(f"Loop({epoch}): {loss}")

plt.plot(loss_list)
plt.title('Training Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model.eval()

with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities > 0.5).float()
    accuracy = (predictions == y_test).float().mean()

print(accuracy)
        